El objetivo es ejecutar el codigo realizado el día miércoles. En la lista de texto, encuentra 5 oraciones que den p_pos = 0.5 y p_neg = 0.5

In [1]:
from transformers import pipeline
import torch 

clf = pipeline(
    'text-classification',
    model='pysentimiento/robertuito-sentiment-analysis',
    tokenizer='pysentimiento/robertuito-sentiment-analysis',
    top_k=None,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [2]:
from datasets import load_dataset
from collections import Counter 
import random 

raw = load_dataset("sepidmnorozy/Spanish_sentiment", split='train')
print(f"Numero de muestras: {len(raw)}. {raw.column_names}, {Counter(raw['label'])}")

Numero de muestras: 1029. ['label', 'text'], Counter({1: 851, 0: 178})


In [3]:
idx = list(range(len(raw)))
random.seed(42)
random.shuffle(idx)

n_val = int(0.2 * len(idx))
val_idx, train_idx = idx[:n_val], idx[n_val:]

train_texts = [raw[i]['text'] for i in train_idx]
train_labels = [int(raw[i]['label']) for i in train_idx]

val_texts = [raw[i]['text'] for i in val_idx]
val_labels = [int(raw[i]['label']) for i in val_idx]

In [4]:
for texto, label in zip(train_texts[:3], train_labels[:3]):
    print(label, texto)

1 Hermoso lugar !
1 Muy correcto en todos los aspectos , muy limpio , tiene detalles que agradan , como el mini · bar gratuito ( bebidas no alcoholicas ) y el calienta liquidos de la habitacion para que te puedas hacer cafe o infusiones.Reponen a diario .
1 En general muy bueno y el barrio donde esta situado es un barrio amable sencillo y de buenas costumbres .


In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_name = "dccuchile/bert-base-spanish-wwm-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: 'Negativo', 1:'Positivo'},
    label2id={'Negativo':0, 'Positivo':1}
).to(device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING

In [6]:
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

class ReviewDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels 

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[i], dtype=torch.long)
        return item

def tokenize_texts(tokenizer, texts, max_length=128):
    return tokenizer(
        texts, 
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors=None
    )

max_length = 128 
batch_size = 16

train_ds = ReviewDataset(
    tokenize_texts(tokenizer, train_texts, max_length),
    train_labels
)

val_ds = ReviewDataset(
    tokenize_texts(tokenizer, val_texts, max_length),
    val_labels
)

In [7]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
n_epochs = 15

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_ds, batch_size=batch_size)

total_steps = len(train_loader) * n_epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1*total_steps),
    num_training_steps=total_steps
)

for epoch in range(n_epochs):
    model.train()
    for batch in train_loader:
        labels = batch.pop('labels').to(device)
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        loss = model(**batch, labels=labels).loss # extraer la perdida
        loss.backward() 

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
        optimizer.step()
        scheduler.step()


In [8]:
from sklearn.metrics import accuracy_score, f1_score

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    for batch in loader:
        labels = batch.pop('labels')
        batch = {k: v.to(device) for k, v in batch.items()}
        preds = model(**batch).logits.argmax(dim=-1).cpu().tolist()
        y_pred.extend(preds)
        y_true.extend(labels.tolist())

    return (
        accuracy_score(y_true, y_pred),
        f1_score(y_true, y_pred, average='macro'),
        y_true,
        y_pred
    )

In [9]:
from sklearn.metrics import classification_report, confusion_matrix

acc, f1, y_true, y_pred = evaluate(model, val_loader, device)

print(f"Accuracy: {acc*100:.2f}%")
print(f"F1-Score: {f1*100:.2f}%")

print(classification_report(
    y_true, y_pred,
    target_names=['Negativo', 'Positivo'],
    digits=2
))

Accuracy: 95.12%
F1-Score: 90.50%
              precision    recall  f1-score   support

    Negativo       0.90      0.79      0.84        33
    Positivo       0.96      0.98      0.97       172

    accuracy                           0.95       205
   macro avg       0.93      0.89      0.90       205
weighted avg       0.95      0.95      0.95       205



In [10]:
@torch.no_grad()
def predecir(textos, tokenizer, model, device, max_length=128):
    model.eval()
    enc = tokenizer(textos, truncation=True, padding=True,
                    max_length=max_length, return_tensors='pt')

    enc = {k: v.to(device) for k, v in enc.items()}
    probs = torch.softmax(model(**enc).logits, dim=-1).cpu() # score que tan seguro esta
    ids = probs.argmax(dim=-1).tolist() # 0 o 1

    return [
        {'texto': t, "label": {0: 'Negativo', 1: 'Positivo'}[i],
         'p_neg': float(p[0]), 'p_pos': float(p[1])}
         for t, i, p in zip(textos, ids, probs)
    ]

In [13]:
textos = [
    "Ponga aqui su texto"
    # TODO: Encontrar 5 oraciones que den p_pos = 0.5 y p_neg = 0.5
]

In [14]:
resultados = predecir(textos, tokenizer, model, device)

In [15]:
for r in resultados:
    print(f"[{r['label'].upper()}] P_NEG: {r['p_neg']:.2f} P_POS: {r['p_pos']:.2f}")
    print(f"    {r['texto']}")

[POSITIVO] P_NEG: 0.00 P_POS: 1.00
    Ponga aqui su texto
